In [59]:
import pandas as pd

In [60]:
df = pd.read_csv("IMDB_dataset.csv")

In [61]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [62]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [63]:
df.duplicated().sum()

np.int64(418)

In [64]:
# DROP duplicated
df.shape
df = df.drop_duplicates()

In [65]:
df.shape

(49582, 2)

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49582 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     49582 non-null  object
 1   sentiment  49582 non-null  object
dtypes: object(2)
memory usage: 1.1+ MB


## Text-Preprocessing

### 1.Converting to lowercase

In [67]:
df["review"] = df["review"].str.lower()

### 2.Removing URLs

In [68]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text
    
df["review"] = df["review"].apply(remove_urls)

### 3.Removing Punctuations

In [69]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4.Removing HTML tags

In [70]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5.Removing the StopWords

In [71]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [72]:
# from nltk.tokenize import word_tokenize
# from nltk.corpus import stopwords

# def removing_stopwords(text):
#     tokens = word_tokenize(text)
#     stop_words = stopwords.words("english")

#     for word in tokens:
#         if word in stop_words:
#             text = text.replace(word, "")
            
#     return text

# df["review"] = df["review"].apply(removing_stopwords)

In [73]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def removing_stopwords(text):
    tokens = word_tokenize(text)

    stop_words = set(stopwords.words("english"))

    filtered_words = []

    for word in tokens:
        if word.lower() not in stop_words:
            filtered_words.append(word)

    return " ".join(filtered_words)

df["review"] = df["review"].apply(removing_stopwords)

In [74]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


### 6.Stemming

In [75]:
from nltk.stem import PorterStemmer

def stemming(text):

    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [76]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


### 7.Encoding

In [77]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [78]:
y = df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8.Vectorization

In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])


In [80]:
X.shape
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4108209 stored elements and shape (49582, 5000)>

### 9.Dataset and Dataloaders

In [81]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [82]:
X_train.shape

(39665, 5000)

In [83]:
X_test.shape

(9917, 5000)

In [84]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = X_train.toarray()
X_test = X_test.toarray()


train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()    
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()  
)

In [85]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=64, shuffle= True)

### 10.Build RNN

In [86]:
import torch.nn as nn
import torch.optim as optim

In [87]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first= True)

        # FC layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out =self.fc(out[:, -1, :])
        return out

In [88]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### 11.Training the RNN

In [89]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1)

        outputs = model(xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        
        
    print(f"FOR epoch {epoch+1}, LOSS = {loss.item()}")
    


FOR epoch 1, LOSS = 0.26059821248054504
FOR epoch 2, LOSS = 0.35117605328559875
FOR epoch 3, LOSS = 0.32535427808761597
FOR epoch 4, LOSS = 0.2805913984775543
FOR epoch 5, LOSS = 0.15724149346351624
FOR epoch 6, LOSS = 0.22259485721588135
FOR epoch 7, LOSS = 0.18590094149112701
FOR epoch 8, LOSS = 0.17312031984329224
FOR epoch 9, LOSS = 0.1763269305229187
FOR epoch 10, LOSS = 0.11446340382099152


### 12.Evaluating our model

In [90]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0

    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)

        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted==yb).sum().item()

    acc = correct_vals/total_vals * 100
    acc = round(acc,2)
    print(f"ACCURACY: {acc}%")
    

ACCURACY: 87.28%
